# DialogSum: GPU pipeline (Colab)

This notebook runs data preparation, the first/last sentence baseline, zero-shot vLLM inference, LoRA training with Unsloth/TRL, LoRA vLLM inference, and BLEU/ROUGE evaluation.

**Before running:** select `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`. Connect Google Drive when prompted. `WANDB_API_KEY` in Colab Secrets is optional; local training logs and a loss plot are always saved.

**No Groq key is needed here.** LLM-as-judge will be run locally by Codex after you return `colab_results.zip`. Do not upload the local `.env`.

If a cell fails, stop and share its error before continuing. GPU compatibility is checked during setup; it has not been tested locally.


In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

def run(command):
    print(f"Running: {command}", flush=True)
    subprocess.run(shlex.split(command), check=True)

run('nvidia-smi')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
os.chdir('/content')
repo = Path('/content/text-generation')
branch = 'feature/lora-summarization'
if not repo.exists():
    run(f'git clone -b {branch} https://github.com/semenovfilipp/text-generation.git')
os.chdir(repo)
# Editable installs in subprocesses may not update the running notebook kernel.
import sys
src_path = str((repo / 'src').resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)
run('git fetch origin')
run(f'git checkout {branch}')
run(f'git pull --ff-only origin {branch}')
run('git rev-parse HEAD')
drive_outputs = Path('/content/drive/MyDrive/text-generation/outputs')
drive_outputs.mkdir(parents=True, exist_ok=True)
outputs = repo / 'outputs'
if outputs.is_symlink():
    assert outputs.resolve() == drive_outputs.resolve(), 'outputs points to another directory'
elif outputs.exists():
    raise RuntimeError('outputs already exists as a directory; move it before rerunning')
else:
    outputs.symlink_to(drive_outputs, target_is_directory=True)


## Environment

Each step runs in a separate process and checks its exit code. Stop at any error before continuing. Installed versions are saved to `outputs/environment.txt`.


In [ ]:
run('python -m pip install -q unsloth vllm')
run('python -m pip install -q -r requirements.txt wandb')
run('python -m pip install -q -e .')
run('python -c "import unsloth; import torch, vllm, trl, peft; print(torch.__version__, vllm.__version__, trl.__version__, peft.__version__)"')
with (outputs / 'environment.txt').open('w') as f:
    subprocess.run(['python', '-m', 'pip', 'freeze'], check=True, stdout=f)


In [ ]:
from google.colab import userdata

# Groq evaluation is performed locally after this notebook finishes.
for name in ['WANDB_API_KEY', 'KAGGLE_USERNAME', 'KAGGLE_KEY', 'HF_TOKEN']:
    try:
        value = userdata.get(name)
        if value:
            os.environ[name] = value
            print(name, 'loaded')
    except Exception:
        print(name, 'not set (optional)')


## Данные и бейзлайн без ML

In [ ]:
run('python scripts/prepare_data.py --config configs/data.yaml')
run('python scripts/baseline_lead.py --config configs/data.yaml')


## Zero-shot Qwen2.5-0.5B-Instruct через vLLM

In [ ]:
run('python scripts/infer.py --config configs/infer_zeroshot.yaml --debug')
run('python scripts/infer.py --config configs/infer_zeroshot.yaml')


## LoRA-дообучение (unsloth + trl SFTTrainer)

Если unsloth не встанет, запасной вариант: `!python scripts/train_lora.py --config configs/train_lora.yaml --backend peft` (чистые trl + peft).

In [ ]:
run('python scripts/train_lora.py --config configs/train_lora.yaml --debug')
run('python scripts/train_lora.py --config configs/train_lora.yaml')


In [ ]:
from IPython.display import Image, display
import json

display(Image('outputs/models/qwen2.5-0.5b-lora/loss.png'))
print(json.load(open('outputs/models/qwen2.5-0.5b-lora/train_summary.json')))

## Инференс LoRA-модели через vLLM

In [ ]:
run('python scripts/infer.py --config configs/infer_lora.yaml')


## Метрики

In [ ]:
run('python scripts/evaluate.py --output outputs/metrics.csv')


In [ ]:
# Make local imports work in this kernel even after an editable pip install.
import sys
from pathlib import Path

project_root = Path(globals().get('repo', Path.cwd())).resolve()
src_path = str(project_root / 'src')
if not (project_root / 'src/dialogsum/utils.py').is_file():
    raise RuntimeError('Project sources not found; run the repository setup cell first')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import pandas as pd
from dialogsum.utils import read_jsonl, write_json

test = {r['id']: r for r in read_jsonl('data/processed/test.jsonl')}
preds = {n: {p['id']: p['prediction'] for p in read_jsonl(f'outputs/predictions/{n}.jsonl')} for n in ['lead_first_last', 'qwen2.5-0.5b_zeroshot', 'qwen2.5-0.5b_lora']}
examples = []
for i in ['test_0', 'test_7', 'test_42', 'test_100', 'test_250']:
    item = {**test[i], 'predictions': {n: p[i] for n, p in preds.items()}, 'review': 'TODO: factual errors, coverage, brevity'}
    examples.append(item)
    print(item)
write_json(examples, 'outputs/manual_review.json')


## Return the results for local judging

GPU work is complete. Groq LLM-as-judge, manual review and final README conclusions will be completed locally by Codex using the predictions in the archive below.

No Groq API calls are made by this notebook. Keep `GROQ_API_KEY` in your local project `.env`.


## Save and download results

The final cell checks that all required results exist, writes `MyDrive/text-generation/colab_results.zip` and starts a browser download. If the download is blocked, download the ZIP directly from Google Drive.

Place the ZIP in your local `text-generation` project folder and tell Codex it is ready. The archive contains predictions, metrics, training logs, a loss plot, environment versions, the exact test split and run configuration; model weights stay on Drive.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files as colab_files
import shutil

# Save the test split and configuration actually used for this run.
shutil.copyfile('data/processed/test.jsonl', outputs / 'test.jsonl')
config_snapshot = outputs / 'run_configs'
shutil.copytree('configs', config_snapshot, dirs_exist_ok=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
write_json({'git_commit': commit, 'branch': branch}, outputs / 'run_manifest.json')

model_dir = outputs / 'models/qwen2.5-0.5b-lora'
required = [outputs / name for name in [
    'metrics.csv', 'environment.txt', 'manual_review.json', 'test.jsonl', 'run_manifest.json',
]]
for system in ['lead_first_last', 'qwen2.5-0.5b_zeroshot', 'qwen2.5-0.5b_lora']:
    required += [outputs / 'predictions' / f'{system}.jsonl',
                 outputs / 'predictions' / f'{system}_meta.json']
required += [model_dir / name for name in ['log_history.json', 'loss.png', 'train_summary.json']]
missing = [str(file) for file in required if not file.is_file()]
if missing:
    raise RuntimeError(f'Missing required results: {missing}')

archive = drive_outputs.parent / 'colab_results.zip'
with ZipFile(archive, 'w', ZIP_DEFLATED) as z:
    for file in required + sorted(config_snapshot.glob('*.yaml')):
        z.write(file, file.relative_to(outputs))
print(f'Results saved: {archive}')
print('Put colab_results.zip in your local text-generation folder and tell Codex it is ready.')
colab_files.download(str(archive))
